# 01 - Download FIRMS Data

This notebook downloads NASA FIRMS active fire detections for Morocco for 2023. The FIRMS area API supports short date ranges, so the year is requested in 5-day chunks. Your real API key is entered at runtime and is not saved in this notebook.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/fire-risk-project")

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project folder does not exist: {PROJECT_DIR}")

DATA_DIR = PROJECT_DIR / "data"
if not DATA_DIR.exists():
    raise FileNotError(f"Data folder does not exist: {DATA_DIR}")

RAW_FIRMS_DIR = PROJECT_DIR / "data" / "raw" / "firms"
RAW_FIRMS_DIR.mkdir(parents=True, exist_ok=True)

FIRMS_OUTPUT_PATH = RAW_FIRMS_DIR / "firms_viirs_morocco_ws_2023.csv"

print("Saving to:", FIRMS_OUTPUT_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving to: /content/drive/MyDrive/fire-risk-project/data/raw/firms/firms_viirs_morocco_ws_2023.csv


In [3]:
import io
import os
import time
from datetime import date, timedelta
from getpass import getpass

import pandas as pd
import requests
from tqdm.auto import tqdm


In [4]:
FIRMS_MAP_KEY = os.getenv("FIRMS_MAP_KEY") or getpass("Enter FIRMS MAP_KEY: ")
assert FIRMS_MAP_KEY, "FIRMS_MAP_KEY is required to download FIRMS data."


Enter FIRMS MAP_KEY: ··········


In [5]:
WEST, SOUTH, EAST, NORTH = -17.2, 20.5, -1.0, 36.2
BBOX = f"{WEST},{SOUTH},{EAST},{NORTH}"
SOURCE = "VIIRS_SNPP_SP"
DAY_RANGE = 5
YEAR = 2023

print("BBOX:", BBOX)
print("SOURCE:", SOURCE)
print("DAY_RANGE:", DAY_RANGE)


BBOX: -17.2,20.5,-1.0,36.2
SOURCE: VIIRS_SNPP_SP
DAY_RANGE: 5


In [6]:
def chunk_start_dates(year, chunk_days=5):
    current = date(year, 1, 1)
    end = date(year, 12, 31)
    while current <= end:
        yield current
        current += timedelta(days=chunk_days)


def build_firms_url(map_key, source, bbox, day_range, start_date):
    return (
        "https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        f"{map_key}/{source}/{bbox}/{day_range}/{start_date:%Y-%m-%d}"
    )


def download_firms_year(year, map_key, source, bbox, day_range=5, sleep_seconds=0.5):
    frames = []
    starts = list(chunk_start_dates(year, chunk_days=day_range))

    for start_date in tqdm(starts, desc=f"Downloading FIRMS {year}"):
        url = build_firms_url(map_key, source, bbox, day_range, start_date)
        try:
            response = requests.get(url, timeout=60)
            response.raise_for_status()
            text = response.text.strip()

            if not text or text.lower().startswith("no"):
                continue

            chunk = pd.read_csv(io.StringIO(text))
            required_cols = {"latitude", "longitude", "acq_date"}
            if not required_cols.issubset(chunk.columns):
                print(f"Skipping {start_date:%Y-%m-%d}: unexpected columns {list(chunk.columns)}")
                continue

            chunk["request_start_date"] = pd.Timestamp(start_date)
            frames.append(chunk)
        except Exception as exc:
            print(f"Skipping {start_date:%Y-%m-%d}: {exc}")

        time.sleep(sleep_seconds)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True).drop_duplicates()


In [7]:
firms_df = download_firms_year(
    year=YEAR,
    map_key=FIRMS_MAP_KEY,
    source=SOURCE,
    bbox=BBOX,
    day_range=DAY_RANGE,
)

firms_df.to_csv(FIRMS_OUTPUT_PATH, index=False)
print(f"Saved {len(firms_df):,} FIRMS rows to {FIRMS_OUTPUT_PATH}")


Saved 4,526 FIRMS rows to /content/drive/MyDrive/fire-risk-project/data/raw/firms/firms_viirs_morocco_ws_2023.csv


In [8]:
print("Shape:", firms_df.shape)
print("Columns:", list(firms_df.columns))

if not firms_df.empty and "acq_date" in firms_df.columns:
    acq_dates = pd.to_datetime(firms_df["acq_date"], errors="coerce")
    print("Date range:", acq_dates.min(), "to", acq_dates.max())
    print("Fire detection count:", len(firms_df))
    display(firms_df.head())
else:
    print("No FIRMS rows were downloaded. Check the MAP_KEY, source, bbox, and API response messages above.")


Shape: (4526, 16)
Columns: ['latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_ti5', 'frp', 'daynight', 'type', 'request_start_date']
Date range: 2023-01-01 00:00:00 to 2023-12-31 00:00:00
Fire detection count: 4526


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type,request_start_date
0,33.84033,-5.63820,332.41,0.45,0.47,2023-01-01,1256,N,VIIRS,n,2,295.43,2.29,D,0,2023-01-01
1,29.40406,-9.01355,336.19,0.57,0.69,2023-01-01,1436,N,VIIRS,n,2,292.25,3.15,D,0,2023-01-01
2,29.40438,-9.01239,337.86,0.57,0.69,2023-01-01,1436,N,VIIRS,n,2,292.39,4.45,D,0,2023-01-01
3,33.55703,-7.52066,300.68,0.66,0.73,2023-01-02,118,N,VIIRS,n,2,279.15,1.64,N,0,2023-01-01
4,34.59477,-2.34885,298.55,0.34,0.56,2023-01-02,118,N,VIIRS,n,2,277.27,0.80,N,2,2023-01-01
